In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

# LINE Bot 設定
line_channel_access_token = "你的 Channel Access Token"
line_channel_secret = "你的 Channel Secret"
port = 5000

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # 取得 LINE 傳來的 X-Line-Signature
    # 這是用來確認訊息是否真的來自 LINE 官方
    signature = request.headers['X-Line-Signature']

    # 取得 LINE 傳來的訊息內容
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # 將 LINE 傳來的訊息交給 WebhookHandler 處理
    try:
        handler.handle(body, signature)

    except InvalidSignatureError:
        app.logger.info(
            "Invalid signature. Please check your channel access token/channel secret."
        )
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    # 取得使用者輸入的文字
    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 判斷使用者輸入是否以 "AI " 開頭
        # 只有以 "AI " 開頭的訊息才會呼叫 Gemini
        #
        # 例如：
        # AI 什麼是 Python
        #
        # 如果沒有以 "AI " 開頭，就不會進入 Gemini 回應流程
        if text.startswith('AI '):

            # 移除前面的 "AI "，只保留使用者真正想問 Gemini 的內容
            #
            # 例如：
            # 使用者輸入：AI 什麼是 Python
            # text[3:] 取得的內容會是：什麼是 Python
            prompt = text[3:]

            # ======================================================
            # 使用 stateful_query() 呼叫 Gemini
            #
            # 為什麼這裡的回應是 stateful？
            #
            # stateful 的意思是「有狀態」或「有記憶上下文」。
            #
            # 也就是說，Gemini 不只會看這一次使用者輸入的 prompt，
            # 還會參考前面已經對話過的內容。
            #
            # 例如：
            # 第一次輸入：AI 什麼是 Python？
            # Gemini 回答：Python 是一種程式語言。
            #
            # 第二次輸入：AI 那它可以做什麼？
            #
            # 因為使用的是 stateful_query()，
            # Gemini 會記得上一句提到的是 Python，
            # 所以它知道「它」指的是 Python。
            #
            # 如果是 stateless_query()，
            # Gemini 只會看到「那它可以做什麼？」
            # 可能會不知道「它」是什麼。
            #
            # 所以這裡使用 stateful_query(prompt)，
            # 是為了讓 Gemini 可以保留上下文，
            # 讓 LINE Bot 像連續聊天一樣回應使用者。
            # ======================================================
            reply_text = stateful_query(prompt)

            # 將 Gemini 產生的回應傳回 LINE 使用者
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=reply_text)
                    ]
                )
            )

        else:
            # 如果不是以 "AI " 開頭，就不會呼叫 Gemini
            # 這裡會直接回覆使用者輸入的文字
            #
            # 因為 messages 裡面放了兩個 TextMessage，
            # 所以 LINE Bot 會回覆兩次一樣的文字
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


if __name__ == "__main__":
    app.run(port=port)